# E-Methanol Reactor Predictive Surrogate Model

This notebook trains a Machine Learning (Random Forest) surrogate model using the synthetic data generated from our rigorous 1D physics-based reactor model. 

It allows for **instantaneous prediction** of reactor performance (CO2 conversion, Methanol Selectivity, and Space-Time Yield) without needing to solve the complex ODEs.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# Load the DOE dataset generated by our 1D model
df = pd.read_csv('../outputs/membrane_doe.csv')
print(f"Loaded {len(df)} simulated reactor cases.")
df.head()

Loaded 500 simulated reactor cases.


## 1. Train the Machine Learning Surrogate Models
We will train a separate Random Forest for each key performance indicator (Target).

In [2]:
features = [
    'T_in_K',
    'P_in_bar',
    'flow_mol_s',
    'h2_co2_ratio'
]

targets = [
    'co2_conversion',
    'meoh_yield'
]

X = df[features]
y = df[targets]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {}

for target in targets:
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train[target])
    models[target] = rf
    
    # Evaluate
    preds = rf.predict(X_test)
    r2 = r2_score(y_test[target], preds)
    print(f"{target}: R² = {r2:.3f}")


co2_conversion: R² = 0.883
meoh_yield: R² = 0.876


## 2. Interactive Prediction Interface
Input your own reactor parameters to instantly predict the production results.

In [3]:
def predict_reactor_performance(T_in, P_in, flow, ratio):
    input_data = pd.DataFrame([{
        'T_in_K': T_in,
        'P_in_bar': P_in,
        'flow_mol_s': flow,
        'h2_co2_ratio': ratio
    }])
    
    print("--- ML Predicted Reactor Performance ---")
    for target, model in models.items():
        pred = model.predict(input_data)[0]
        if 'sty' in target:
            print(f"Methanol Production (STY): {pred:.4f} kg/(m³·h)")
        elif 'conversion' in target:
            print(f"CO2 Conversion:            {pred*100:.2f}%")
        elif 'yield' in target:
            print(f"Methanol Yield:            {pred*100:.2f}%")

# --- TEST THE PREDICTOR HERE ---
predict_reactor_performance(
    T_in=493.15,      # 220 °C
    P_in=50.0,        # bar
    flow=0.015,       # mol/s
    ratio=3.5         # H2:CO2 ratio
)

--- ML Predicted Reactor Performance ---
CO2 Conversion:            6.14%
Methanol Yield:            5.58%


## 3. Feature Importance Analysis
Let's see which operating conditions have the biggest impact on Methanol Production.

In [4]:
target = 'meoh_yield'
importances = models[target].feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 5))
plt.title("Impact of Operating Conditions on Methanol Production")
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

## 4. Predictive Model Performance (Predicted vs Actual)
Let's visualize how accurately our Machine Learning models predict the true rigorous physics simulations. Points closer to the dashed diagonal line indicate perfect predictions.

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, target in zip(axes, targets):
    y_pred = models[target].predict(X_test)
    ax.scatter(y_test[target], y_pred, alpha=0.6, color='seagreen', edgecolor='k')
    
    # Diagonal reference line
    min_val = min(y_test[target].min(), y_pred.min())
    max_val = max(y_test[target].max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='Perfect Prediction')
    
    ax.set_title(f'{target.replace("_", " ").title()}')
    ax.set_xlabel('Actual (1D Physics Simulation)')
    ax.set_ylabel('Predicted (Random Forest)')
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()
